# Лабораторная работа 3. Доверительные интервалы и проверка гипотез

**Набор данных:** [Weight loss using diet](https://www.kaggle.com/datasets/tombenny/foodhabbits).
78 человек были распределены на одну из трёх диет; вес измерялся до диеты и через 6 недель.

Признаки: `Person` — номер участника, `gender` — пол (0 — женщина, 1 — мужчина), `Age` — возраст,
`Height` — рост, `pre.weight` — вес до диеты (кг), `Diet` — тип диеты (1, 2, 3), `weight6weeks` — вес через 6 недель (кг).

**Исследуемый признак:** потеря веса за 6 недель `weight_loss = pre.weight − weight6weeks` (кг).

## 1. Загрузка и первичная обработка данных

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

df = pd.read_csv('foodDiet.csv')
print(df.shape)
df.head()

(78, 7)


,Person,gender,Age,Height,pre.weight,Diet,weight6weeks
0,25,NaN,41,171,60,2,60.0
1,26,NaN,32,174,103,2,103.0
2,1,0.0,22,159,58,1,54.2
3,2,0.0,46,192,60,1,54.0
4,3,0.0,55,170,64,1,63.3


In [3]:
print('Пропуски по столбцам:')
print(df.isna().sum())
df[df.isna().any(axis=1)]

Пропуски по столбцам:
Person          0
gender          2
Age             0
Height          0
pre.weight      0
Diet            0
weight6weeks    0
dtype: int64


,Person,gender,Age,Height,pre.weight,Diet,weight6weeks
0,25,NaN,41,171,60,2,60.0
1,26,NaN,32,174,103,2,103.0


Пропущен пол у двух участников (№ 25 и 26). При этом их вес через 6 недель в точности равен начальному, то есть итоговое измерение, скорее всего, не проводилось (участники выбыли). Заполнять такие строки нет смысла, поэтому удаляем их. Остаётся 76 наблюдений.

### Основные статистики

In [4]:
x = df['weight_loss'].values
n = len(x)

print(f'Среднее:                  {np.mean(x):.3f}')
print(f'Медиана:                  {np.median(x):.3f}')
print(f'Стандартное отклонение:   {np.std(x, ddof=1):.3f}')
print(f'Квартили Q1, Q3:          {np.percentile(x, 25):.3f}, {np.percentile(x, 75):.3f}')
print(f'Минимум, максимум:        {x.min():.3f}, {x.max():.3f}')
print('\nСреднее по типам диеты:')
print(df.groupby('Diet')['weight_loss'].agg(['count', 'mean', 'std']).round(3))

KeyError: 'weight_loss'

### Визуализация

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(x, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(np.mean(x), color='black', label=f'Среднее = {np.mean(x):.2f}')
axes[0].axvline(np.median(x), color='orange', linestyle='--', label=f'Медиана = {np.median(x):.2f}')
axes[0].set_title('Гистограмма потери веса')
axes[0].set_xlabel('Потеря веса, кг')
axes[0].legend()

axes[1].boxplot(x)
axes[1].set_title('Ящик с усами')
axes[1].set_ylabel('Потеря веса, кг')

stats.probplot(x, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q график')
axes[2].set_xlabel('Теоретические квантили')
axes[2].set_ylabel('Квантили выборки')

plt.tight_layout()
plt.show()

Средняя потеря веса — 3.95 кг, медиана — 3.70 кг. Они близки, значит распределение почти симметричное. Выбросов на ящике с усами нет. Точки на Q-Q графике лежат практически на прямой, поэтому распределение можно считать близким к нормальному.

## 2. Параметрические доверительные интервалы

In [ ]:
def compute_ci(data, alpha=0.05):
    # ДИ для среднего на основе t-распределения
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    return mean - t_crit * se, mean + t_crit * se

def variance_ci(data, alpha=0.05):
    # ДИ для дисперсии на основе χ²-распределения
    n = len(data)
    s2 = np.var(data, ddof=1)
    return ((n - 1) * s2 / stats.chi2.ppf(1 - alpha / 2, n - 1),
            (n - 1) * s2 / stats.chi2.ppf(alpha / 2, n - 1))

ci95 = compute_ci(x, 0.05)
ci99 = compute_ci(x, 0.01)
var95 = variance_ci(x, 0.05)

print(f'95% ДИ для среднего:   ({ci95[0]:.3f}, {ci95[1]:.3f})')
print(f'99% ДИ для среднего:   ({ci99[0]:.3f}, {ci99[1]:.3f})')
print(f'95% ДИ для дисперсии:  ({var95[0]:.3f}, {var95[1]:.3f}),  s² = {np.var(x, ddof=1):.3f}')

## 3. Бутстреп-доверительные интервалы

In [ ]:
def bootstrap_ci(data, statistic, n_bootstrap=5000, ci=95):
    estimates = []
    n = len(data)
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        estimates.append(statistic(sample))
    estimates = np.array(estimates)

    # Процентильный интервал
    lower = (100 - ci) / 2
    perc_ci = np.percentile(estimates, [lower, 100 - lower])

    # Нормальный интервал: оценка по исходной выборке ± z * SE_boot
    se_boot = np.std(estimates, ddof=1)
    z = stats.norm.ppf(1 - (1 - ci / 100) / 2)
    theta = statistic(data)
    normal_ci = (theta - z * se_boot, theta + z * se_boot)

    return normal_ci, perc_ci, estimates

normal_mean, perc_mean, boot_means = bootstrap_ci(x, np.mean)
_, perc_median, boot_medians = bootstrap_ci(x, np.median)

print(f'Среднее, бутстреп-нормальный 95% ДИ:     ({normal_mean[0]:.3f}, {normal_mean[1]:.3f})')
print(f'Среднее, бутстреп-процентильный 95% ДИ:  ({perc_mean[0]:.3f}, {perc_mean[1]:.3f})')
print(f'Медиана, бутстреп-процентильный 95% ДИ:  ({perc_median[0]:.3f}, {perc_median[1]:.3f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].hist(boot_means, bins=40, density=True, color='skyblue', alpha=0.7)
axes[0].axvline(np.mean(x), color='black', label='Исходное среднее')
axes[0].axvline(perc_mean[0], color='red', linestyle='--', label='Процентильный 95% ДИ')
axes[0].axvline(perc_mean[1], color='red', linestyle='--')
axes[0].set_title('Бутстреп-распределение среднего')
axes[0].set_xlabel('Средняя потеря веса, кг')
axes[0].legend()

axes[1].hist(boot_medians, bins=40, density=True, color='plum', alpha=0.7)
axes[1].axvline(np.median(x), color='black', label='Исходная медиана')
axes[1].axvline(perc_median[0], color='red', linestyle='--', label='Процентильный 95% ДИ')
axes[1].axvline(perc_median[1], color='red', linestyle='--')
axes[1].set_title('Бутстреп-распределение медианы')
axes[1].set_xlabel('Медиана потери веса, кг')
axes[1].legend()

plt.tight_layout()
plt.show()

### Сравнение интервалов

In [ ]:
comparison = pd.DataFrame({
    'Метод': ['Параметрический (t)', 'Бутстреп нормальный', 'Бутстреп процентильный', 'Медиана: бутстреп процентильный'],
    'Нижняя': [ci95[0], normal_mean[0], perc_mean[0], perc_median[0]],
    'Верхняя': [ci95[1], normal_mean[1], perc_mean[1], perc_median[1]],
})
comparison['Ширина'] = comparison['Верхняя'] - comparison['Нижняя']
comparison.round(3)

Все интервалы для среднего практически совпадают. Причина в том, что распределение потери веса близко к нормальному, и при n = 76 распределение выборочного среднего тоже нормально.

Интервал для медианы (3.1; 4.3) кг сдвинут влево вслед за самой медианой. Его распределение неровное, потому что медиана может принимать только значения из исходной выборки. Для медианы простой параметрической формулы нет, и здесь бутстреп особенно полезен.

## 4. Проверка гипотез

Уровень значимости α = 0.05. 

### 4.1. Одновыборочный t-тест

Рекомендуемый безопасный темп похудения — около 0.5 кг в неделю, т.е. 3 кг за 6 недель.

* $H_0: \mu = 3$ — средняя потеря веса равна 3 кг
* $H_1: \mu \ne 3$

In [ ]:
t1, p1 = stats.ttest_1samp(x, 3)
d1 = (np.mean(x) - 3) / np.std(x, ddof=1)
print(f't = {t1:.4f}, df = {n - 1}, p = {p1:.4f}')
print(f"Cohen's d = {d1:.3f}")
print(f'Вывод: {"отвергаем H0" if p1 < 0.05 else "не отвергаем H0"}')

p = 0.0015 < 0.05, поэтому отвергаем H₀: средняя потеря веса значимо больше 3 кг.ё При этом Cohen's d = 0.38, то есть эффект малый.

### 4.2. Двухвыборочный t-тест

Сравним потерю веса на диете 3 и диете 1.

* $H_0: \mu_3 = \mu_1$
* $H_1: \mu_3 \ne \mu_1$

In [ ]:
diet1 = df.loc[df['Diet'] == 1, 'weight_loss']
diet3 = df.loc[df['Diet'] == 3, 'weight_loss']

_, p_levene = stats.levene(diet3, diet1)
equal_var = p_levene > 0.05
t2, p2 = stats.ttest_ind(diet3, diet1, equal_var=equal_var)
d2 = (diet3.mean() - diet1.mean()) / np.sqrt((diet3.var() + diet1.var()) / 2)

print(f'Диета 3: n = {len(diet3)}, среднее = {diet3.mean():.3f}')
print(f'Диета 1: n = {len(diet1)}, среднее = {diet1.mean():.3f}')
print(f'Тест Левена: p = {p_levene:.4f} -> equal_var = {equal_var}')
print(f't = {t2:.4f}, p = {p2:.4f}')
print(f"Cohen's d = {d2:.3f}")
print(f'Вывод: {"различия статистически значимы" if p2 < 0.05 else "нет значимых различий"}')

sns.boxplot(data=df, x='Diet', y='weight_loss', palette='Set2')
plt.title('Потеря веса по типам диеты')
plt.xlabel('Диета'); plt.ylabel('Потеря веса, кг')
plt.show()

### 4.3. Парный t-тест

Вес каждого участника измерен до и после диеты — выборки зависимые.

* $H_0: \mu_{до} = \mu_{после}$ — вес не изменился
* $H_1: \mu_{до} \ne \mu_{после}$

In [ ]:
before = df['pre.weight'].values
after = df['weight6weeks'].values
diffs = before - after

t3, p3 = stats.ttest_rel(before, after)
d3 = np.mean(diffs) / np.std(diffs, ddof=1)
print(f'Среднее до = {before.mean():.2f}, после = {after.mean():.2f}')
print(f't = {t3:.4f}, p = {p3:.3e}')
print(f"Cohen's d = {d3:.3f}")
print(f'Вывод: {"диета дала значимый эффект" if p3 < 0.05 else "нет значимого эффекта"}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(before, after, alpha=0.6)
axes[0].plot([55, 90], [55, 90], 'r--', label='y = x')
axes[0].set_xlabel('Вес до, кг'); axes[0].set_ylabel('Вес после, кг')
axes[0].set_title('Парные данные'); axes[0].legend()
axes[1].hist(diffs, bins=15, color='green', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--', label='нулевая разность')
axes[1].set_title('Распределение разностей (до − после)'); axes[1].legend()
plt.tight_layout()
plt.show()

### 4.4. Критерий хи-квадрат

Клинически значимым считается снижение веса на 5% и более. Связан ли тип диеты с достижением этого результата?

* $H_0$: тип диеты и достижение снижения >= 5% независимы
* $H_1$: между ними есть связь

In [ ]:





df['success'] = (df['weight_loss'] / df['pre.weight'] * 100 >= 5).map({True: '≥ 5%', False: '< 5%'})
contingency = pd.crosstab(df['Diet'], df['success'])
print(contingency)

chi2, p4, dof, expected = stats.chi2_contingency(contingency)
cramer_v = stats.contingency.association(contingency, method='cramer')
print('\nОжидаемые частоты:\n', expected.round(1))
print(f'\nχ^2 = {chi2:.4f}, df = {dof}, p = {p4:.4f}')
print(f'Вывод: {"есть связь" if p4 < 0.05 else "связи нет"}')

p = 0.013 < 0.05: тип диеты связан с достижением снижения веса ≥ 5%. 

## 5. Выводы

**Что показывают ДИ и p-значения.** За 6 недель участники потеряли в среднем 3.95 кг (95% ДИ: 3.37–4.52 кг). Снижение веса статистически значимо (парный t-тест) и немного превышает рекомендуемые 3 кг. Диета 3 эффективнее диеты 1 в среднем на 1.85 кг, и на ней больше доля участников, сбросивших ≥ 5% веса.

**Влияет ли метод на выводы.** Нет. Параметрические и бутстреп-интервалы для среднего почти совпадают, потому что данные близки к нормальным. Бутстреп даёт интервалы немного уже, но ни один вывод от этого не меняется. Бутстреп действительно нужен только для медианы, для которой нет простой формулы.

**Наиболее адекватный тест.** Парный t-тест. Данные собраны по схеме «до/после» на одних и тех же людях, разности распределены нормально, а эффект очень большой.